# Análisis de tráfico aéreo y condiciones ambientales con sensores y APIs en la nube
**CPD – Proyecto 3ª Evaluación** · Cómputo paralelo y distribuido

📦 **Repositorio GitHub:** https://github.com/Anngel-o/trafico-aereo-computo-paralelo
📓 **Notebook en Colab:** https://colab.research.google.com/drive/1a8DVBhEwQmpc0ToJ4koVQ1SSI4rfTell?usp=sharing

**Equipo:**
- Jose Angel Ortiz Meraz — 353195
- Luz Mariam Garcia Castillo — 348409
- Carlos Sánchez Fuentes — 368058


Arquitectura (según el enunciado): **APIs en la nube → asyncio → MPI4Py → CUDA/Numba/CuPy → Dashboard**, con **Pandas** como apoyo.

| Fase | Tecnología | Qué hace aquí |
|---|---|---|
| 1 | `asyncio` + `aiohttp` | Descarga concurrente de AirLabs, OpenWeather, OpenAQ y ThingSpeak |
| 2 | `pandas` | Limpieza y unión de tablas |
| 3 | `mpi4py` | Reparte aeropuertos entre procesos (bootstrap del retraso medio) |
| 4 | `numba.cuda` + `cupy` | Distancias, atípicos, rutas simuladas y correlaciones en GPU |
| 5 | `matplotlib` | Gráficas y respuestas a las preguntas de análisis |

**Antes de ejecutar:** Entorno de ejecución → *Cambiar tipo de entorno* → **GPU T4**. Opcional: agrega tus API keys en 🔑 *Secretos* (`AIRLABS_KEY`, `OPENWEATHER_KEY`, `OPENAQ_KEY`). Sin key, esa fuente usa **datos SIMULADOS** (columna `fuente = SIMULADO`).

## 0. Preparación del entorno (MPI + GPU)

In [ ]:
# Instala OpenMPI (el "motor" de MPI) y el paquete de Python mpi4py que lo controla
!apt-get install -y -qq libopenmpi-dev openmpi-bin > /dev/null
!pip install -q mpi4py aiohttp
# Verifica que Colab nos dio una GPU NVIDIA
!nvidia-smi -L

## 1. Módulos del proyecto
Cada celda guarda un archivo `.py` (`%%writefile`) para poder importarlo y subirlo tal cual a GitHub.

In [ ]:
%%writefile config.py
"""
config.py
=========
Configuración central del proyecto "Análisis de tráfico aéreo y condiciones
ambientales" (CPD - 3ª Evaluación).

Aquí vive TODO lo que se comparte entre módulos:
  * la lista de aeropuertos que vamos a analizar (con sus coordenadas),
  * la caja geográfica (bounding box) de México para pedir vuelos a AirLabs,
  * la función que obtiene las API keys SIN escribirlas en el código.

Idea para exponer: separar la configuración del código evita repetir datos
y evita subir llaves secretas a GitHub.

REQUISITOS PREVIOS (cuentas y API keys)
---------------------------------------
Antes de ejecutar con datos reales hay que crear cuentas y generar llaves:

  * AirLabs      -> cuenta en airlabs.co            -> AIRLABS_KEY      (formato UUID)
  * OpenWeather  -> cuenta en openweathermap.org    -> OPENWEATHER_KEY  (32 hexadecimales;
                    las llaves nuevas pueden tardar un rato en activarse)
  * OpenAQ v3    -> registro en explore.openaq.org/register; la llave se ve en
                    explore.openaq.org/account     -> OPENAQ_KEY
  * ThingSpeak   -> NO necesita cuenta ni llave aquí (ver nota abajo).

Las llaves se guardan en Colab -> candado 🔑 (Secretos), con el interruptor
"Acceso desde el cuaderno" ACTIVADO. Nunca se escriben en el código.
El módulo verificar_claves.py comprueba que cada llave exista, tenga el formato
esperado y sea aceptada por su API.
"""
import os

# ---------------------------------------------------------------------------
# 1) Aeropuertos de México que analizaremos.
#    Formato:  código IATA -> (nombre, latitud, longitud)
#    Las coordenadas las usamos para pedirle a OpenWeather/OpenAQ el clima y
#    la calidad del aire justo en el aeropuerto.
# ---------------------------------------------------------------------------
AEROPUERTOS = {
    "MEX": ("CDMX Benito Juárez", 19.4363, -99.0721),
    "CUN": ("Cancún",             21.0365, -86.8771),
    "GDL": ("Guadalajara",        20.5218, -103.3112),
    "MTY": ("Monterrey",          25.7785, -100.1069),
    "TIJ": ("Tijuana",            32.5411, -116.9700),
    "CUU": ("Chihuahua",          28.7029, -105.9647),
    "CJS": ("Ciudad Juárez",      31.6361, -106.4287),
    "PVR": ("Puerto Vallarta",    20.6801, -105.2544),
    "SJD": ("Los Cabos",          23.1518, -109.7215),
    "MID": ("Mérida",             20.9370, -89.6577),
    "BJX": ("Del Bajío",          20.9935, -101.4808),
    "HMO": ("Hermosillo",         29.0959, -111.0479),
}

# Caja que cubre México: (lat_sur, lon_oeste, lat_norte, lon_este).
# AirLabs la pide en ese orden dentro del parámetro "bbox".
BBOX_MEXICO = (14.5, -118.5, 32.8, -86.5)

# Canal PÚBLICO de ThingSpeak: la estación meteorológica de MathWorks.
# ¿Por qué NO hace falta crear cuenta ni generar API key?
#   ThingSpeak organiza los datos en "canales". Un canal marcado como PÚBLICO se
#   puede leer con un simple GET a
#       https://api.thingspeak.com/channels/12397/feeds.json
#   sin autenticación (acceso gratuito de solo lectura). Las "API keys" de
#   ThingSpeak (Read/Write API Key) solo se necesitan para:
#     - leer un canal PRIVADO, o
#     - ESCRIBIR datos (publicar lecturas de nuestros propios sensores).
#   Como aquí solo leemos datos ambientales públicos (temperatura, humedad,
#   presión, viento...), usamos el canal 12397 tal cual.
THINGSPEAK_CANAL = 12397

# Carpetas de salida
DIR_DATOS = "datos"
DIR_RESULTADOS = "resultados"
DIR_FIGURAS = "figuras"


def obtener_clave(nombre, limpiar=True):
    """
    Busca una API key sin escribirla en el código.

    Orden de búsqueda:
      1. Secretos de Google Colab (el candado 🔑 de la barra izquierda).
      2. Variable de entorno (útil si lo corres en tu computadora).
    Devuelve None si no la encuentra -> el sistema entra en MODO DEMO
    para esa fuente (datos simulados, claramente etiquetados).

    limpiar=True quita espacios, saltos de línea y comillas al inicio/final: el
    error más común al copiar y pegar una llave es llevarse un espacio o un
    salto de línea invisible, y eso hace que la API la rechace.
    """
    valor = None
    try:
        from google.colab import userdata          # solo existe en Colab
        valor = userdata.get(nombre)
    except Exception:
        pass                                        # no estamos en Colab o no hay secreto
    if not valor:
        valor = os.environ.get(nombre)
    if valor and limpiar:
        valor = valor.strip().strip("\"'").strip()
    return valor or None


In [ ]:
%%writefile datos_demo.py
"""
datos_demo.py
=============
Generador de datos SIMULADOS (modo demo).

¿Para qué sirve? Si todavía no tienes una API key (AirLabs, OpenWeather,
OpenAQ) o la cuota se acabó, el proyecto sigue corriendo con datos sintéticos
que tienen EXACTAMENTE el mismo formato (columnas) que los reales.

IMPORTANTE (honestidad en el reporte): estos datos se marcan con la columna
`fuente = "SIMULADO"`. Las relaciones clima->retraso que aparecen aquí las
pusimos nosotros a propósito, así que NO son un hallazgo científico; sirven
para probar el pipeline. Los resultados reales salen al usar las API keys.
"""
import numpy as np
import pandas as pd
from config import AEROPUERTOS, BBOX_MEXICO

RNG = np.random.default_rng(42)       # semilla fija -> resultados reproducibles
AEROLINEAS = ["AM", "Y4", "VB", "VW", "AA", "UA", "DL", "AV"]


def clima_demo():
    """Un renglón de clima por aeropuerto (mismas columnas que OpenWeather)."""
    filas = []
    for iata, (_, lat, lon) in AEROPUERTOS.items():
        lluvia = max(0.0, RNG.normal(0.4, 1.2))               # mm en la última hora
        filas.append(dict(
            iata=iata, temp=RNG.normal(26, 6), humedad=RNG.uniform(25, 90),
            presion=RNG.normal(1012, 5), viento_vel=abs(RNG.normal(4, 2.5)),
            viento_dir=RNG.uniform(0, 360),
            visibilidad_m=float(np.clip(10000 - lluvia * 1500 + RNG.normal(0, 800), 800, 10000)),
            lluvia_mm=lluvia, nubosidad=RNG.uniform(0, 100),
            condicion="Rain" if lluvia > 0.5 else "Clear", fuente="SIMULADO"))
    return pd.DataFrame(filas)


def calidad_aire_demo():
    """PM2.5 y PM10 por aeropuerto (mismas columnas que OpenAQ)."""
    filas = []
    for iata in AEROPUERTOS:
        pm25 = abs(RNG.normal(18, 9))
        filas.append(dict(iata=iata, pm25=pm25, pm10=pm25 * RNG.uniform(1.4, 2.2),
                          estacion="demo", distancia_m=RNG.uniform(1000, 20000),
                          fuente="SIMULADO"))
    return pd.DataFrame(filas)


def horarios_demo(clima, n_por_aeropuerto=50):
    """
    Salidas programadas con retrasos. El retraso depende (a propósito) de la
    lluvia, la visibilidad y el viento del aeropuerto + ruido aleatorio.
    """
    filas = []
    for _, c in clima.iterrows():
        base = (3 + 12 * c.lluvia_mm + 0.004 * (10000 - c.visibilidad_m)
                + 1.5 * c.viento_vel)
        for _ in range(n_por_aeropuerto):
            hora = int(np.clip(RNG.normal(13, 4.5), 0, 23))   # más salidas a mediodía
            retraso = max(0.0, RNG.gamma(1.6, base / 1.6))
            if RNG.random() < 0.01:                            # algunos atípicos extremos
                retraso += RNG.uniform(150, 400)
            cancelado = RNG.random() < 0.015 + 0.01 * c.lluvia_mm
            filas.append(dict(
                flight_iata=f"{RNG.choice(AEROLINEAS)}{RNG.integers(100, 999)}",
                airline_iata=RNG.choice(AEROLINEAS), dep_iata=c.iata,
                arr_iata=RNG.choice([a for a in AEROPUERTOS if a != c.iata]),
                status="cancelled" if cancelado else "scheduled",
                dep_delayed=np.nan if cancelado else retraso, arr_delayed=np.nan,
                dep_time=f"2026-11-23 {hora:02d}:{RNG.integers(0, 59):02d}",
                fuente="SIMULADO"))
    return pd.DataFrame(filas)


def vuelos_demo(n=800):
    """Vuelos 'en el aire' con posición, altitud (m) y velocidad (km/h)."""
    s, w, nn, e = BBOX_MEXICO
    codigos = list(AEROPUERTOS)
    df = pd.DataFrame(dict(
        flight_iata=[f"{RNG.choice(AEROLINEAS)}{RNG.integers(100, 999)}" for _ in range(n)],
        airline_iata=RNG.choice(AEROLINEAS, n),
        dep_iata=RNG.choice(codigos, n), arr_iata=RNG.choice(codigos, n),
        lat=RNG.uniform(s, nn, n), lng=RNG.uniform(w, e, n),
        alt=np.clip(RNG.normal(9500, 2500, n), 300, 13000),
        speed=np.clip(RNG.normal(780, 120, n), 150, 1000),
        dir=RNG.uniform(0, 360, n), v_speed=RNG.normal(0, 3, n),
        status="en-route", aircraft_icao=RNG.choice(["B738", "A320", "A321", "B789"], n),
        fuente="SIMULADO"))
    k = max(3, n // 100)                       # inyectamos atípicos de velocidad/altitud
    idx = RNG.choice(n, k, replace=False)
    df.loc[idx, "speed"] = RNG.uniform(1400, 1800, k)
    df.loc[idx, "alt"] = RNG.uniform(17000, 22000, k)
    return df


def thingspeak_demo(n=500):
    """Serie de tiempo de un sensor ambiental (mismo formato que ThingSpeak)."""
    t = pd.date_range(end=pd.Timestamp.now(tz="UTC"), periods=n, freq="1min")
    ciclo = np.sin(np.linspace(0, 6 * np.pi, n))
    return pd.DataFrame(dict(
        created_at=t, temperatura=22 + 4 * ciclo + RNG.normal(0, .3, n),
        humedad=55 - 12 * ciclo + RNG.normal(0, 1, n),
        presion=30.0 + .05 * ciclo + RNG.normal(0, .01, n),
        velocidad_viento=abs(3 + 2 * ciclo + RNG.normal(0, .8, n)),
        fuente="SIMULADO"))


In [ ]:
%%writefile ingesta_async.py
"""
ingesta_async.py
================
FASE 1 de la arquitectura: "Python + Asyncio -> lectura concurrente de APIs".

Concepto clave para exponer
---------------------------
Pedir datos a una API es tiempo de ESPERA (la red tarda), no de cálculo.
Con código normal (secuencial) esperamos una respuesta antes de mandar la
siguiente petición. Con `asyncio` mandamos muchas peticiones "al mismo tiempo"
y mientras una espera, el programa atiende a las otras. Es concurrencia
(no paralelismo de CPU): un solo hilo, muchas esperas traslapadas.

Piezas que usamos:
  * `async def`      -> define una "corrutina" (función que puede pausarse).
  * `await`          -> "pausa aquí hasta que llegue la respuesta, y mientras
                         tanto deja que otras corrutinas avancen".
  * `asyncio.gather` -> lanza varias corrutinas juntas y junta sus resultados.
  * `Semaphore`      -> límite de peticiones simultáneas (para no saturar ni
                         violar el rate-limit de las APIs gratuitas).
  * `aiohttp`        -> cliente HTTP asíncrono (equivalente async de `requests`).
"""
import asyncio
import time

import aiohttp
import pandas as pd

import datos_demo
from config import (AEROPUERTOS, BBOX_MEXICO, THINGSPEAK_CANAL, obtener_clave)

AIRLABS = "https://airlabs.co/api/v9"
OWM = "https://api.openweathermap.org/data/2.5/weather"
OPENAQ = "https://api.openaq.org/v3"
THINGSPEAK = "https://api.thingspeak.com/channels"
ULTIMA_INGESTA_S = None          # segundos que tardó la última ingesta (lo usa el reporte PDF)


# ---------------------------------------------------------------------------
# Función base: UNA petición GET con límite de concurrencia y reintentos
# ---------------------------------------------------------------------------
async def pedir_json(sesion, url, params=None, headers=None, sem=None, reintentos=2):
    """
    Hace GET a `url` y devuelve el JSON (o None si falla).

    - `sem` (Semaphore): `async with sem` deja pasar solo N peticiones a la vez.
    - Reintentamos con una pausa creciente (backoff) si la red falla.
    - Devolver None (en vez de lanzar error) permite que UNA API caída no
      tumbe todo el pipeline: después caemos a modo demo para esa fuente.
    """
    for intento in range(reintentos + 1):
        try:
            async with sem:
                async with sesion.get(url, params=params, headers=headers,
                                      timeout=aiohttp.ClientTimeout(total=25)) as r:
                    if r.status == 200:
                        datos = await r.json(content_type=None)
                        # AirLabs (y otras) a veces responden HTTP 200 pero con un
                        # campo "error" (ej. llave incorrecta): lo tratamos como fallo.
                        if isinstance(datos, dict) and datos.get("error"):
                            print(f"  ! {url} -> la API rechazó la petición: {datos['error']}")
                            return None
                        return datos
                    print(f"  ! {url.split('?')[0]} -> HTTP {r.status}"
                          + (" (llave rechazada o sin activar)" if r.status in (401, 403) else ""))
                    if r.status in (401, 403, 404):    # reintentar no ayuda con estos errores
                        return None
        except Exception as e:                      # timeout, DNS, etc.
            print(f"  ! error de red ({type(e).__name__}) en {url}")
        await asyncio.sleep(1.5 * (intento + 1))    # backoff antes de reintentar
    return None


# ---------------------------------------------------------------------------
# AirLabs: vuelos en el aire y horarios (retrasos)
# ---------------------------------------------------------------------------
async def obtener_vuelos(sesion, sem, clave):
    """Todos los vuelos activos dentro de la caja de México (endpoint /flights)."""
    bbox = ",".join(str(x) for x in BBOX_MEXICO)
    datos = await pedir_json(sesion, f"{AIRLABS}/flights",
                             {"api_key": clave, "bbox": bbox}, sem=sem)
    if not datos or "response" not in datos:
        return None
    df = pd.DataFrame(datos["response"])
    df["fuente"] = "AirLabs"
    return df


async def obtener_horarios(sesion, sem, clave, iata):
    """Salidas programadas de UN aeropuerto con retraso en minutos (/schedules)."""
    datos = await pedir_json(sesion, f"{AIRLABS}/schedules",
                             {"api_key": clave, "dep_iata": iata, "limit": 50}, sem=sem)
    if not datos or "response" not in datos:
        return None
    df = pd.DataFrame(datos["response"])
    df["fuente"] = "AirLabs"
    return df


# ---------------------------------------------------------------------------
# OpenWeather: clima actual en cada aeropuerto
# ---------------------------------------------------------------------------
async def obtener_clima(sesion, sem, clave, iata, lat, lon):
    """Clima actual en las coordenadas del aeropuerto (unidades métricas)."""
    d = await pedir_json(sesion, OWM,
                         {"lat": lat, "lon": lon, "appid": clave, "units": "metric"}, sem=sem)
    if not d:
        return None
    return dict(
        iata=iata, temp=d["main"]["temp"], humedad=d["main"]["humidity"],
        presion=d["main"]["pressure"], viento_vel=d["wind"].get("speed", 0),
        viento_dir=d["wind"].get("deg", 0), visibilidad_m=d.get("visibility", 10000),
        lluvia_mm=d.get("rain", {}).get("1h", 0.0),       # el campo solo existe si llueve
        nubosidad=d["clouds"]["all"], condicion=d["weather"][0]["main"], fuente="OpenWeather")


# ---------------------------------------------------------------------------
# OpenAQ: calidad del aire (PM2.5 / PM10) cerca del aeropuerto
# ---------------------------------------------------------------------------
async def obtener_calidad_aire(sesion, sem, clave, iata, lat, lon, radio_m=25000):
    """
    1) Busca estaciones a `radio_m` metros del aeropuerto (/locations).
    2) Pide la última medición de la más cercana que mida PM2.5 (/latest).
    """
    h = {"X-API-Key": clave}                              # OpenAQ v3 usa este header
    locs = await pedir_json(sesion, f"{OPENAQ}/locations",
                            {"coordinates": f"{lat},{lon}", "radius": radio_m, "limit": 10},
                            headers=h, sem=sem)
    if not locs or not locs.get("results"):
        return None
    for loc in locs["results"]:                           # ya vienen ordenadas por cercanía
        sensores = {s["id"]: s["parameter"]["name"] for s in loc.get("sensors", [])}
        if "pm25" not in sensores.values():
            continue
        ult = await pedir_json(sesion, f"{OPENAQ}/locations/{loc['id']}/latest",
                               headers=h, sem=sem)
        if not ult:
            continue
        medidas = {sensores.get(m["sensorsId"]): m["value"] for m in ult.get("results", [])}
        return dict(iata=iata, pm25=medidas.get("pm25"), pm10=medidas.get("pm10"),
                    estacion=loc.get("name"), distancia_m=loc.get("distance"),
                    fuente="OpenAQ")
    return None


# ---------------------------------------------------------------------------
# ThingSpeak: sensores ambientales (canal público, sin API key)
# ---------------------------------------------------------------------------
async def obtener_thingspeak(sesion, sem, canal=THINGSPEAK_CANAL, resultados=500):
    """
    Lee las últimas `resultados` lecturas del canal. Los nombres de los campos
    (field1..field8) se leen del propio canal, no los suponemos.

    NOTA: aquí NO se manda ninguna API key. El canal 12397 es PÚBLICO (estación
    meteorológica de MathWorks) y ThingSpeak permite leer canales públicos con un
    GET sin autenticación. La key de ThingSpeak solo se necesita para canales
    privados o para escribir datos (ver config.py).
    """
    d = await pedir_json(sesion, f"{THINGSPEAK}/{canal}/feeds.json",
                         {"results": resultados}, sem=sem)
    if not d:
        return None
    nombres = {f"field{i}": d["channel"].get(f"field{i}") for i in range(1, 9)
               if d["channel"].get(f"field{i}")}
    df = pd.DataFrame(d["feeds"]).rename(columns=nombres)
    df["created_at"] = pd.to_datetime(df["created_at"])
    for col in nombres.values():
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["fuente"] = f"ThingSpeak-{canal}"
    return df.drop(columns=["entry_id"], errors="ignore")


# ---------------------------------------------------------------------------
# Orquestador: lanza TODAS las peticiones a la vez con asyncio.gather
# ---------------------------------------------------------------------------
async def ingestar_todo(max_concurrentes=8):
    """
    Descarga todo en paralelo (concurrente) y devuelve un diccionario de
    DataFrames: vuelos, horarios, clima, aire, thingspeak.
    Si falta una API key o la API falla -> usa datos DEMO de esa fuente.
    """
    global ULTIMA_INGESTA_S
    t0 = time.perf_counter()
    k_air, k_owm, k_oaq = (obtener_clave(n) for n in
                           ("AIRLABS_KEY", "OPENWEATHER_KEY", "OPENAQ_KEY"))
    sem = asyncio.Semaphore(max_concurrentes)
    async with aiohttp.ClientSession() as s:
        # Se arman las "tareas" (todavía no se ejecutan)...
        t_ts = obtener_thingspeak(s, sem)
        t_vuelos = obtener_vuelos(s, sem, k_air) if k_air else asyncio.sleep(0)
        t_hor = [obtener_horarios(s, sem, k_air, i) if k_air else asyncio.sleep(0)
                 for i in AEROPUERTOS]
        t_clima = [obtener_clima(s, sem, k_owm, i, la, lo) if k_owm else asyncio.sleep(0)
                   for i, (_, la, lo) in AEROPUERTOS.items()]
        t_aire = [obtener_calidad_aire(s, sem, k_oaq, i, la, lo) if k_oaq else asyncio.sleep(0)
                  for i, (_, la, lo) in AEROPUERTOS.items()]
        # ...y aquí se lanzan TODAS juntas. `await` espera a que terminen todas.
        ts, vuelos, *resto = await asyncio.gather(t_ts, t_vuelos, *t_hor, *t_clima, *t_aire)

    n = len(AEROPUERTOS)
    hor, clima, aire = resto[:n], resto[n:2 * n], resto[2 * n:]

    # Armamos cada DataFrame; si no hay datos reales, caemos a demo.
    hor = [h for h in hor if isinstance(h, pd.DataFrame)]
    horarios = pd.concat(hor, ignore_index=True) if hor else None
    clima = [c for c in clima if isinstance(c, dict)]
    clima = pd.DataFrame(clima) if clima else None
    aire = [a for a in aire if isinstance(a, dict)]
    aire = pd.DataFrame(aire) if aire else None
    vuelos = vuelos if isinstance(vuelos, pd.DataFrame) else None

    clima_final = clima if clima is not None else datos_demo.clima_demo()
    out = dict(
        vuelos=vuelos if vuelos is not None else datos_demo.vuelos_demo(),
        horarios=horarios if horarios is not None else datos_demo.horarios_demo(clima_final),
        clima=clima_final,
        aire=aire if aire is not None else datos_demo.calidad_aire_demo(),
        thingspeak=ts if ts is not None else datos_demo.thingspeak_demo(),
    )
    ULTIMA_INGESTA_S = time.perf_counter() - t0
    print(f"Ingesta terminada en {ULTIMA_INGESTA_S:.2f} s")
    for nombre, df in out.items():
        if df is None:
            print(f"  {nombre:11s}: SIN DATOS")
        else:
            print(f"  {nombre:11s}: {len(df):5d} filas | fuente = {df['fuente'].iloc[0]}")
    return out


# ---------------------------------------------------------------------------
# Demostración didáctica: secuencial vs asyncio (sin gastar cuota de APIs)
# ---------------------------------------------------------------------------
async def demo_asyncio_vs_secuencial(n_peticiones=12, latencia=0.5):
    """
    Simula `n_peticiones` que tardan `latencia` s cada una (asyncio.sleep imita
    la espera de red). Secuencial ~ n*latencia; asyncio ~ latencia.
    Devuelve (t_secuencial, t_asyncio).
    """
    t0 = time.perf_counter()
    for _ in range(n_peticiones):
        await asyncio.sleep(latencia)               # una tras otra
    t_sec = time.perf_counter() - t0

    t0 = time.perf_counter()
    await asyncio.gather(*[asyncio.sleep(latencia) for _ in range(n_peticiones)])
    t_async = time.perf_counter() - t0              # todas a la vez
    return t_sec, t_async


if __name__ == "__main__":
    # Fuera de un notebook se usa asyncio.run; en Colab basta `await ingestar_todo()`.
    datos = asyncio.run(ingestar_todo())


In [ ]:
%%writefile limpieza_pandas.py
"""
limpieza_pandas.py
==================
FASE de apoyo (la barra morada de la arquitectura): limpieza, transformación y
unificación de estructuras con PANDAS.

Pandas es la librería de tablas (DataFrames). Aquí hacemos 4 cosas típicas:
  1. convertir columnas a número (`pd.to_numeric`),
  2. quitar duplicados y filas inservibles,
  3. crear columnas nuevas (región, hora, bandera de retraso),
  4. unir tablas (`merge`) para cruzar retrasos con clima.
"""
import numpy as np
import pandas as pd


def limpiar_vuelos(df):
    """Vuelos en el aire: tipos numéricos, rangos físicos razonables y región."""
    df = df.copy()
    for c in ["lat", "lng", "alt", "speed", "dir", "v_speed"]:
        if c not in df:                       # el plan gratis de AirLabs omite algunos campos
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.drop_duplicates(subset=["flight_iata", "lat", "lng"])
    df = df.dropna(subset=["lat", "lng"])                      # sin posición no sirve
    # NOTA: NO borramos velocidades/alturas raras aquí; se detectan como ATÍPICOS
    # más adelante (GPU). Solo quitamos lo físicamente imposible (negativos).
    df = df[(df["speed"].fillna(0) >= 0) & (df["alt"].fillna(0) >= 0)]
    # Región por latitud (simple, suficiente para el análisis)
    df["region"] = pd.cut(df["lat"], bins=[-90, 19, 25, 90],
                          labels=["Sur-Sureste", "Centro", "Norte"])
    return df.reset_index(drop=True)


def limpiar_horarios(df):
    """Salidas programadas: retrasos numéricos, banderas de cancelado/retrasado, hora."""
    df = df.copy()
    for c in ["dep_delayed", "arr_delayed"]:
        if c not in df:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["cancelado"] = df["status"].eq("cancelled")
    df["retrasado"] = df["dep_delayed"].fillna(0) > 15          # >15 min = retraso (criterio común)
    df["dep_delayed"] = df["dep_delayed"].fillna(0).clip(lower=0)  # adelantos -> 0
    df["hora"] = pd.to_datetime(df["dep_time"], errors="coerce").dt.hour
    return df.drop_duplicates(subset=["flight_iata", "dep_iata", "dep_time"]).reset_index(drop=True)


def resumen_por_aeropuerto(horarios, clima, aire):
    """
    Tabla final: UNA fila por aeropuerto con retrasos + clima + calidad del aire.
    `groupby` agrupa los vuelos por aeropuerto y `agg` calcula estadísticas;
    luego `merge` pega el clima y el aire usando el código IATA como llave.
    """
    r = (horarios.groupby("dep_iata")
         .agg(n_vuelos=("flight_iata", "size"),
              retraso_medio=("dep_delayed", "mean"),
              retraso_p90=("dep_delayed", lambda x: np.percentile(x, 90)),
              pct_retrasados=("retrasado", "mean"),
              cancelados=("cancelado", "sum"))
         .reset_index().rename(columns={"dep_iata": "iata"}))
    r["pct_retrasados"] *= 100
    r = r.merge(clima.drop(columns="fuente"), on="iata", how="left")
    r = r.merge(aire[["iata", "pm25", "pm10"]], on="iata", how="left")
    return r


def vuelos_por_hora(horarios):
    """Cuántas salidas hay en cada hora del día (para ver el ciclo diario)."""
    return horarios.groupby("hora").size().reindex(range(24), fill_value=0)


In [ ]:
%%writefile mpi_analisis.py
"""
mpi_analisis.py
===============
FASE 2 de la arquitectura: "MPI4Py -> distribución del procesamiento".

Se ejecuta con:   mpirun -n 4 python mpi_analisis.py
(mpirun lanza 4 COPIAS de este mismo script; cada copia se llama "rank".)

Concepto clave para exponer
---------------------------
MPI = Message Passing Interface. Cada proceso tiene su PROPIA memoria; para
compartir datos se mandan mensajes. El patrón que usamos es el clásico:

      rank 0 reparte  --scatter-->  todos calculan su parte  --gather--> rank 0 junta

  * comm.Get_rank()  -> mi número de proceso (0, 1, 2...)
  * comm.Get_size()  -> cuántos procesos hay en total
  * comm.bcast       -> rank 0 manda el MISMO dato a todos (parámetros)
  * comm.scatter     -> rank 0 manda un PEDAZO DISTINTO a cada proceso
  * comm.gather      -> todos mandan su resultado a rank 0
  * comm.Reduce      -> combina arreglos de todos (aquí: suma) en rank 0

Qué calculamos: por aeropuerto, estadísticas de retraso + un intervalo de
confianza al 95% del retraso medio con BOOTSTRAP (remuestrear miles de veces).
El bootstrap es costoso en CPU y cada aeropuerto es independiente, así que es
un problema ideal para repartir entre procesos.
"""
import json
import os
import socket
import sys
import time

import numpy as np
import pandas as pd
from mpi4py import MPI

import limpieza_pandas as L


def bootstrap_ic95(x, n_boot, rng):
    """
    Intervalo de confianza 95% de la media de `x` por bootstrap:
    1) remuestrea con reemplazo `n_boot` veces, 2) saca la media de cada
    remuestreo, 3) toma los percentiles 2.5 y 97.5 de esas medias.
    """
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))     # índices al azar
    medias = x[idx].mean(axis=1)
    return np.percentile(medias, [2.5, 97.5])


def main():
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    n_boot = int(sys.argv[1]) if len(sys.argv) > 1 else 200_000

    # --- 1) Solo rank 0 lee el archivo y prepara el reparto ------------------
    if rank == 0:
        horarios = L.limpiar_horarios(pd.read_csv("datos/horarios.csv"))
        aeropuertos = sorted(horarios["dep_iata"].unique())
        # Reparto "round-robin": aeropuerto i -> proceso (i % size). Así la carga
        # queda equilibrada aunque haya más aeropuertos que procesos.
        trozos = [horarios[horarios["dep_iata"].isin(aeropuertos[r::size])]
                  for r in range(size)]
    else:
        trozos, n_boot = None, None

    comm.Barrier()                         # todos esperan para medir tiempo parejo
    t0 = MPI.Wtime()

    # --- 2) Distribución de datos -------------------------------------------
    n_boot = comm.bcast(n_boot if rank == 0 else None, root=0)   # mismo parámetro a todos
    mio = comm.scatter(trozos, root=0)                           # un pedazo distinto a cada uno

    # --- 3) Cálculo LOCAL (cada proceso trabaja solo con sus aeropuertos) ----
    rng = np.random.default_rng(1234 + rank)       # semilla distinta por proceso
    filas = []
    for iata, g in mio.groupby("dep_iata"):
        x = g["dep_delayed"].to_numpy(float)
        lo, hi = bootstrap_ic95(x, n_boot, rng)
        filas.append(dict(iata=iata, n=len(x), media=x.mean(), mediana=np.median(x),
                          ic95_inf=lo, ic95_sup=hi, rank_que_calculo=rank))
    horas_local = np.bincount(mio["hora"].dropna().astype(int), minlength=24).astype("int64")
    t_local = MPI.Wtime() - t0
    print(f"[rank {rank}/{size}] host={socket.gethostname()} aeropuertos="
          f"{len(filas)} filas={len(mio)} tiempo_local={t_local:.2f}s", flush=True)

    # --- 4) Agregación de resultados ----------------------------------------
    todas = comm.gather(filas, root=0)                  # lista de listas -> rank 0
    horas_total = np.zeros(24, dtype="int64") if rank == 0 else None
    comm.Reduce(horas_local, horas_total, op=MPI.SUM, root=0)   # suma de histogramas
    t_total = MPI.Wtime() - t0

    if rank == 0:
        df = pd.DataFrame([f for sub in todas for f in sub]).sort_values("media", ascending=False)
        os.makedirs("resultados", exist_ok=True)
        df.to_csv("resultados/mpi_estadisticas.csv", index=False)
        pd.Series(horas_total, name="salidas").rename_axis("hora").to_csv("resultados/mpi_salidas_por_hora.csv")
        # Guardamos el tiempo de esta corrida para comparar 1, 2, 4 procesos
        ruta = "resultados/mpi_tiempos.json"
        tiempos = json.load(open(ruta)) if os.path.exists(ruta) else {}
        tiempos[str(size)] = t_total
        json.dump(tiempos, open(ruta, "w"))
        print(f"\nTOTAL con {size} proceso(s): {t_total:.2f} s (bootstrap={n_boot:,})")
        print(df.round(1).to_string(index=False))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile gpu_numba.py
"""
gpu_numba.py
============
FASE 3 de la arquitectura: "CUDA / CuPy / Numba -> aceleración de cálculos".

Concepto clave para exponer
---------------------------
Una GPU tiene miles de núcleos pequeños. Si la MISMA operación se repite sobre
millones de datos independientes (distancias, z-scores...), la GPU lo hace en
paralelo masivo. En CUDA:

  * KERNEL  = función que corre en la GPU (aquí con el decorador @cuda.jit).
  * HILO (thread) = una copia del kernel; cada hilo procesa UN elemento.
  * BLOQUE (block) = grupo de hilos (usamos 256). GRID = conjunto de bloques.
  * cuda.grid(1)  = "¿qué número de hilo global soy?" -> es el índice i del dato.
  * Hay que COPIAR datos CPU->GPU (host->device) y regresar el resultado;
    esas copias cuestan tiempo, por eso la GPU solo gana con muchos datos.

Usamos:
  * Numba (@cuda.jit)   -> escribimos los kernels nosotros en Python.
  * CuPy                -> como NumPy pero en la GPU (media, desv. est., correlación).

Si no hay GPU (o no está instalado CuPy) todo cae automáticamente a NumPy, así
el código nunca se rompe: solo es más lento.
"""
import math
import time

import numpy as np

try:
    from numba import cuda
    HAY_GPU = cuda.is_available()
except Exception:                     # numba sin soporte CUDA
    cuda, HAY_GPU = None, False

try:
    import cupy as cp
except Exception:
    cp = None

R_TIERRA_KM = 6371.0
HILOS = 256                           # hilos por bloque (múltiplo de 32 = tamaño de un "warp")


# ===========================================================================
# KERNELS (corren en la GPU). Solo se definen si Numba + CUDA existen.
# ===========================================================================
if cuda is not None:

    @cuda.jit
    def k_haversine(lat1, lon1, lat2, lon2, out):
        """
        Distancia de círculo máximo (km) entre dos puntos de la Tierra.
        Cada hilo i calcula la distancia del par i -> out[i].
        """
        i = cuda.grid(1)                              # mi índice global
        if i < out.size:                              # el último bloque puede sobrar
            p1 = lat1[i] * math.pi / 180.0            # grados -> radianes
            p2 = lat2[i] * math.pi / 180.0
            dphi = p2 - p1
            dlam = (lon2[i] - lon1[i]) * math.pi / 180.0
            a = (math.sin(dphi / 2.0) ** 2
                 + math.cos(p1) * math.cos(p2) * math.sin(dlam / 2.0) ** 2)
            out[i] = 2.0 * 6371.0 * math.asin(math.sqrt(a))

    @cuda.jit
    def k_zscore(x, media, desv, umbral, flags):
        """
        Detección de atípicos: z = (x - media)/desv. Si |z| > umbral -> atípico (1).
        """
        i = cuda.grid(1)
        if i < x.size:
            z = (x[i] - media) / desv
            flags[i] = 1 if abs(z) > umbral else 0

    @cuda.jit
    def k_ruta(lat1, lon1, lat2, lon2, pasos, out_lat, out_lon):
        """
        Simulación de rutas: para cada vuelo i y cada paso j calcula el punto
        intermedio sobre el círculo máximo (interpolación esférica).
        Usa una malla 2D: eje x = vuelo, eje y = paso.
        """
        i, j = cuda.grid(2)
        if i < lat1.size and j < pasos:
            t = j / (pasos - 1.0)                     # fracción del recorrido: 0 .. 1
            f1 = lat1[i] * math.pi / 180.0
            l1 = lon1[i] * math.pi / 180.0
            f2 = lat2[i] * math.pi / 180.0
            l2 = lon2[i] * math.pi / 180.0
            # distancia angular entre origen y destino
            d = 2.0 * math.asin(math.sqrt(math.sin((f2 - f1) / 2.0) ** 2 + math.cos(f1)
                                          * math.cos(f2) * math.sin((l2 - l1) / 2.0) ** 2))
            if d < 1e-9:                              # origen == destino
                A, B = 1.0 - t, t
            else:
                A = math.sin((1.0 - t) * d) / math.sin(d)
                B = math.sin(t * d) / math.sin(d)
            x = A * math.cos(f1) * math.cos(l1) + B * math.cos(f2) * math.cos(l2)
            y = A * math.cos(f1) * math.sin(l1) + B * math.cos(f2) * math.sin(l2)
            z = A * math.sin(f1) + B * math.sin(f2)
            out_lat[i, j] = math.atan2(z, math.sqrt(x * x + y * y)) * 180.0 / math.pi
            out_lon[i, j] = math.atan2(y, x) * 180.0 / math.pi


# ===========================================================================
# Versiones CPU (NumPy): sirven de comparación y de plan B sin GPU
# ===========================================================================
def haversine_cpu(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R_TIERRA_KM * np.arcsin(np.sqrt(a))


def zscore_cpu(x, umbral=3.0):
    return (np.abs((x - x.mean()) / x.std()) > umbral).astype(np.int32)


# ===========================================================================
# Envoltorios (host): copian datos a la GPU, lanzan el kernel y regresan
# ===========================================================================
def _bloques(n):
    """Cuántos bloques de HILOS hilos hacen falta para cubrir n datos."""
    return (n + HILOS - 1) // HILOS


def haversine(lat1, lon1, lat2, lon2):
    """Distancias en km. Usa la GPU si existe; si no, NumPy."""
    if not HAY_GPU:
        return haversine_cpu(*(np.asarray(a, float) for a in (lat1, lon1, lat2, lon2)))
    d = [cuda.to_device(np.asarray(a, dtype=np.float32)) for a in (lat1, lon1, lat2, lon2)]  # CPU->GPU
    out = cuda.device_array(len(lat1), dtype=np.float32)                                     # memoria en GPU
    k_haversine[_bloques(len(lat1)), HILOS](*d, out)       # lanzamiento: [bloques, hilos](args)
    return out.copy_to_host()                              # GPU->CPU


def detectar_atipicos(x, umbral=3.0):
    """Devuelve un arreglo de 0/1 (1 = atípico) usando z-score."""
    x = np.asarray(x, dtype=np.float32)
    x = np.nan_to_num(x, nan=float(np.nanmean(x)))         # NaN -> media (no cuenta como atípico)
    if not HAY_GPU:
        return zscore_cpu(x, umbral)
    if cp is not None:                                     # media y desv. est. en GPU con CuPy
        xd = cp.asarray(x); media, desv = float(xd.mean()), float(xd.std())
    else:
        media, desv = float(x.mean()), float(x.std())
    if desv == 0:
        return np.zeros(len(x), dtype=np.int32)
    flags = cuda.device_array(len(x), dtype=np.int32)
    k_zscore[_bloques(len(x)), HILOS](cuda.to_device(x), np.float32(media),
                                      np.float32(desv), np.float32(umbral), flags)
    return flags.copy_to_host()


def rutas_simuladas(lat1, lon1, lat2, lon2, pasos=50):
    """Matriz (n_vuelos x pasos) de latitudes y longitudes a lo largo de cada ruta."""
    n = len(lat1)
    if not HAY_GPU:                                        # versión NumPy vectorizada
        t = np.linspace(0, 1, pasos)[None, :]
        f1, l1, f2, l2 = (np.radians(np.asarray(a, float))[:, None] for a in (lat1, lon1, lat2, lon2))
        d = 2 * np.arcsin(np.sqrt(np.sin((f2 - f1) / 2) ** 2 + np.cos(f1) * np.cos(f2) * np.sin((l2 - l1) / 2) ** 2))
        d = np.where(d < 1e-9, 1e-9, d)
        A, B = np.sin((1 - t) * d) / np.sin(d), np.sin(t * d) / np.sin(d)
        x = A * np.cos(f1) * np.cos(l1) + B * np.cos(f2) * np.cos(l2)
        y = A * np.cos(f1) * np.sin(l1) + B * np.cos(f2) * np.sin(l2)
        z = A * np.sin(f1) + B * np.sin(f2)
        return np.degrees(np.arctan2(z, np.hypot(x, y))), np.degrees(np.arctan2(y, x))
    d = [cuda.to_device(np.asarray(a, dtype=np.float32)) for a in (lat1, lon1, lat2, lon2)]
    o_lat = cuda.device_array((n, pasos), dtype=np.float32)
    o_lon = cuda.device_array((n, pasos), dtype=np.float32)
    bloque = (16, 16)                                      # 16x16 = 256 hilos por bloque (malla 2D)
    grid = ((n + 15) // 16, (pasos + 15) // 16)
    k_ruta[grid, bloque](*d, pasos, o_lat, o_lon)
    return o_lat.copy_to_host(), o_lon.copy_to_host()


def matriz_correlacion(df, columnas, metodo="pearson"):
    """
    Matriz de correlación entre columnas. En GPU con CuPy (cp.corrcoef).
    Spearman = Pearson sobre los rangos (más robusto a atípicos).
    """
    datos = df[columnas].dropna()
    if metodo == "spearman":
        datos = datos.rank()
    m = datos.to_numpy(float).T                            # corrcoef pide variables en filas
    r = cp.asnumpy(cp.corrcoef(cp.asarray(m))) if (HAY_GPU and cp is not None) else np.corrcoef(m)
    import pandas as pd
    return pd.DataFrame(r, index=columnas, columns=columnas)


# ===========================================================================
# Benchmark CPU vs GPU
# ===========================================================================
def benchmark(n=5_000_000, repeticiones=3):
    """
    Compara haversine en NumPy (CPU) vs kernel CUDA (GPU) con n pares de puntos.
    Mide dos cosas en GPU: solo el kernel y el total (con copias CPU<->GPU).
    """
    rng = np.random.default_rng(0)
    lat1, lat2 = rng.uniform(14, 33, n).astype(np.float32), rng.uniform(14, 33, n).astype(np.float32)
    lon1, lon2 = rng.uniform(-118, -86, n).astype(np.float32), rng.uniform(-118, -86, n).astype(np.float32)

    t = time.perf_counter()
    for _ in range(repeticiones):
        ref = haversine_cpu(lat1, lon1, lat2, lon2)
    t_cpu = (time.perf_counter() - t) / repeticiones
    res = dict(n=n, cpu_s=t_cpu, gpu_disponible=bool(HAY_GPU))
    if not HAY_GPU:
        return res

    haversine(lat1[:1000], lon1[:1000], lat2[:1000], lon2[:1000])   # "calentamiento": compila el kernel (JIT)
    t = time.perf_counter()
    for _ in range(repeticiones):
        out = haversine(lat1, lon1, lat2, lon2)                      # incluye copias
    res["gpu_total_s"] = (time.perf_counter() - t) / repeticiones

    d = [cuda.to_device(a) for a in (lat1, lon1, lat2, lon2)]        # copias ANTES de medir
    o = cuda.device_array(n, dtype=np.float32)
    cuda.synchronize(); t = time.perf_counter()                      # synchronize: las llamadas a GPU son asíncronas
    for _ in range(repeticiones):
        k_haversine[_bloques(n), HILOS](*d, o)
    cuda.synchronize()
    res["gpu_kernel_s"] = (time.perf_counter() - t) / repeticiones
    res["max_error_km"] = float(np.max(np.abs(out - ref)))           # verificación de que ambos coinciden
    res["speedup_total"] = t_cpu / res["gpu_total_s"]
    res["speedup_kernel"] = t_cpu / res["gpu_kernel_s"]
    return res


if __name__ == "__main__":
    print("GPU disponible:", HAY_GPU)
    print(benchmark())


In [ ]:
%%writefile analisis.py
"""
analisis.py
===========
Une todas las piezas: usa pandas (limpieza), la GPU (atípicos, distancias,
rutas, correlación) y responde las 5 PREGUNTAS DE ANÁLISIS del proyecto.
"""
import numpy as np
import pandas as pd

import gpu_numba as g
from config import AEROPUERTOS

VARS_CLIMA = ["temp", "humedad", "presion", "viento_vel", "visibilidad_m", "lluvia_mm", "nubosidad"]


def region_de_lat(lat):
    """Misma regla de región que usamos para los vuelos (por latitud)."""
    return "Norte" if lat >= 25 else ("Centro" if lat >= 19 else "Sur-Sureste")


def marcar_atipicos(vuelos, horarios, umbral=3.0):
    """
    Detección de valores atípicos (z-score > umbral) EN GPU en:
    velocidad y altitud de vuelos, y retraso de salidas.
    """
    v, h = vuelos.copy(), horarios.copy()
    for col in ["speed", "alt"]:
        v[f"atip_{col}"] = g.detectar_atipicos(v[col], umbral) if v[col].notna().any() else 0
    h["atip_retraso"] = g.detectar_atipicos(h["dep_delayed"], umbral)
    return v, h


def vuelos_cerca_de_aeropuertos(vuelos, radio_km=150):
    """
    Cuenta cuántos vuelos activos hay a menos de `radio_km` de cada aeropuerto.
    Es una matriz (vuelos x aeropuertos) de distancias -> trabajo ideal para GPU:
    aplanamos todas las parejas y mandamos UN solo kernel.
    """
    n, m = len(vuelos), len(AEROPUERTOS)
    la = np.repeat(vuelos["lat"].to_numpy(float), m)           # cada vuelo repetido m veces
    lo = np.repeat(vuelos["lng"].to_numpy(float), m)
    ap_la = np.tile([a[1] for a in AEROPUERTOS.values()], n)   # los m aeropuertos repetidos n veces
    ap_lo = np.tile([a[2] for a in AEROPUERTOS.values()], n)
    d = g.haversine(la, lo, ap_la, ap_lo).reshape(n, m)
    return pd.Series((d < radio_km).sum(axis=0), index=list(AEROPUERTOS), name="vuelos_cercanos")


def resumen_vuelos(v):
    """Vuelos activos, velocidad y altitud promedio por región."""
    return (v.groupby("region", observed=True)
            .agg(vuelos=("flight_iata", "size"), velocidad_media_kmh=("speed", "mean"),
                 altitud_media_m=("alt", "mean")).round(1))


def rutas_mas_frecuentes(v, top=10):
    """Pares origen-destino más repetidos."""
    r = v.dropna(subset=["dep_iata", "arr_iata"])
    return (r.assign(ruta=r["dep_iata"] + " → " + r["arr_iata"])["ruta"]
            .value_counts().head(top))


def simular_rutas(v, n_max=40, pasos=50):
    """Simula la trayectoria origen->destino de hasta n_max vuelos (GPU)."""
    ok = v[v["dep_iata"].isin(AEROPUERTOS) & v["arr_iata"].isin(AEROPUERTOS)
           & (v["dep_iata"] != v["arr_iata"])].head(n_max)
    c = lambda col, i: [AEROPUERTOS[x][i] for x in ok[col]]
    la, lo = g.rutas_simuladas(c("dep_iata", 1), c("dep_iata", 2), c("arr_iata", 1), c("arr_iata", 2), pasos)
    return ok.reset_index(drop=True), la, lo


def tabla_final(resumen, cercanos):
    """Agrega región y tráfico cercano a la tabla por aeropuerto."""
    t = resumen.copy()
    t["region"] = t["iata"].map(lambda i: region_de_lat(AEROPUERTOS[i][1]))
    t["vuelos_cercanos"] = t["iata"].map(cercanos)
    return t


def _r(t, a, b, metodo="spearman"):
    """Correlación entre dos columnas (NaN si alguna no varía)."""
    d = t[[a, b]].dropna()
    return np.nan if d[a].nunique() < 2 or d[b].nunique() < 2 else d[a].corr(d[b], method=metodo)


def responder_preguntas(t, h):
    """
    Calcula y redacta la respuesta a cada pregunta del proyecto.
    Devuelve una lista de textos (también se guarda en el reporte).
    """
    out = []
    # 1) ¿Relación clima-retrasos? Spearman por aeropuerto (n = nº de aeropuertos)
    rs = {v: _r(t, v, "retraso_medio") for v in VARS_CLIMA}
    mejor = max(rs, key=lambda k: abs(rs[k]) if not np.isnan(rs[k]) else -1)
    out.append("1) ¿Hay relación entre clima y retrasos? Correlación de Spearman con el retraso medio "
               f"(n={len(t)} aeropuertos): " + ", ".join(f"{k}={v:+.2f}" for k, v in rs.items())
               + f". La más fuerte: {mejor} ({rs[mejor]:+.2f}). Con tan pocos aeropuertos solo "
               "|r|>0.58 sería significativa (p<0.05).")
    # 2) Lluvia o baja visibilidad: comparamos retrasos de vuelos en aeropuertos con/sin condición adversa
    hh = h.merge(t[["iata", "lluvia_mm", "visibilidad_m"]], left_on="dep_iata", right_on="iata")
    adverso = (hh["lluvia_mm"] > 0.5) | (hh["visibilidad_m"] < 8000)
    if adverso.any() and (~adverso).any():
        a, b = hh.loc[adverso, "dep_delayed"], hh.loc[~adverso, "dep_delayed"]
        out.append(f"2) ¿Aumentan los retrasos con lluvia (>0.5 mm/h) o visibilidad <8 km? Retraso medio "
                   f"con condición adversa: {a.mean():.1f} min (n={len(a)}) vs. normal: {b.mean():.1f} min "
                   f"(n={len(b)}); diferencia {a.mean() - b.mean():+.1f} min.")
    else:
        out.append("2) No hubo aeropuertos con condición adversa en esta muestra: no se puede comparar.")
    # 3) Ranking de aeropuertos
    top = t.sort_values("retraso_medio", ascending=False).head(3)
    out.append("3) Aeropuertos con mayor retraso medio: " + "; ".join(
        f"{r.iata} ({r.retraso_medio:.1f} min, {r.pct_retrasados:.0f}% >15 min)" for r in top.itertuples()) + ".")
    # 4) Actividad por hora y región
    hh2 = h.dropna(subset=["hora"]).copy()
    hh2["region"] = hh2["dep_iata"].map(lambda i: region_de_lat(AEROPUERTOS[i][1]))
    pico = hh2.groupby("region")["hora"].agg(lambda s: int(s.value_counts().idxmax()))
    out.append("4) Hora con más salidas por región: " + ", ".join(f"{r}={int(x):02d}:00 h" for r, x in pico.items())
               + f". En todo el país el pico es a las {int(hh2['hora'].value_counts().idxmax()):02d}:00 h.")
    # 5) Condiciones ambientales vs tráfico cercano
    cols = ["temp", "humedad", "viento_vel", "lluvia_mm", "pm25"]
    r5 = {c: _r(t, c, "vuelos_cercanos") for c in cols}
    out.append("5) Condiciones que coinciden con más tráfico aéreo (vuelos a <150 km, Spearman): "
               + ", ".join(f"{k}={v:+.2f}" for k, v in r5.items()) + ".")
    return out


In [ ]:
%%writefile visualizacion.py
"""
visualizacion.py
================
FASE 4 (Dashboard / Reportes): gráficas con Matplotlib. Cada función guarda un
PNG en figuras/ y lo devuelve para mostrarlo en el notebook.

Colores: paleta Okabe-Ito (segura para daltonismo). Usamos pocos colores y
etiquetas directas para que las gráficas se entiendan al exponer.
"""
import os

import matplotlib.pyplot as plt
import numpy as np

from config import AEROPUERTOS, DIR_FIGURAS

AZUL, NARANJA, VERDE, ROJO, GRIS = "#0072B2", "#E69F00", "#009E73", "#D55E00", "#666666"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110, "font.size": 10})


def _guardar(fig, nombre):
    os.makedirs(DIR_FIGURAS, exist_ok=True)
    fig.tight_layout()
    fig.savefig(f"{DIR_FIGURAS}/{nombre}.png", dpi=130)
    return fig


def fig_actividad_vuelos(v, rutas):
    """Vuelos por región, por aerolínea y rutas más frecuentes."""
    f, ax = plt.subplots(1, 3, figsize=(15, 4))
    v["region"].value_counts().plot.bar(ax=ax[0], color=AZUL, rot=0); ax[0].set_title("Vuelos activos por región")
    v["airline_iata"].value_counts().head(10).plot.bar(ax=ax[1], color=NARANJA, rot=0)
    ax[1].set_title("Vuelos por aerolínea (top 10)")
    rutas.iloc[::-1].plot.barh(ax=ax[2], color=VERDE); ax[2].set_title("Rutas más frecuentes")
    return _guardar(f, "01_actividad_vuelos")


def fig_retrasos_aeropuerto(t, mpi_df):
    """Retraso medio por aeropuerto con intervalo de confianza 95% (calculado con MPI)."""
    d = mpi_df.sort_values("media")
    f, ax = plt.subplots(figsize=(8, 5))
    err = np.vstack([d["media"] - d["ic95_inf"], d["ic95_sup"] - d["media"]])
    ax.barh(d["iata"], d["media"], xerr=err, color=AZUL, ecolor=GRIS, capsize=3)
    ax.set_xlabel("Retraso medio de salida (min) con IC 95% bootstrap (MPI)")
    ax.set_title("Retrasos por aeropuerto")
    return _guardar(f, "02_retrasos_aeropuerto")


def fig_clima_vs_retraso(t):
    """Dispersión clima vs retraso medio, un punto por aeropuerto, con recta de ajuste."""
    vars_ = [("lluvia_mm", "Lluvia (mm/h)"), ("visibilidad_m", "Visibilidad (m)"), ("viento_vel", "Viento (m/s)")]
    f, ax = plt.subplots(1, 3, figsize=(15, 4))
    for a, (c, nombre) in zip(ax, vars_):
        a.scatter(t[c], t["retraso_medio"], color=AZUL, s=40)
        for r in t.itertuples():
            a.annotate(r.iata, (getattr(r, c), r.retraso_medio), fontsize=8, xytext=(3, 3), textcoords="offset points")
        if t[c].nunique() > 1:
            m, b = np.polyfit(t[c], t["retraso_medio"], 1)
            xs = np.linspace(t[c].min(), t[c].max(), 10); a.plot(xs, m * xs + b, color=ROJO)
            a.set_title(f"{nombre}  (Pearson r={t[c].corr(t['retraso_medio']):+.2f})")
        a.set_xlabel(nombre); a.set_ylabel("Retraso medio (min)")
    return _guardar(f, "03_clima_vs_retraso")


def fig_actividad_hora(h, region_de_lat):
    """Salidas por hora del día, total y por región."""
    d = h.dropna(subset=["hora"]).copy()
    d["region"] = d["dep_iata"].map(lambda i: region_de_lat(AEROPUERTOS[i][1]))
    tab = d.groupby(["hora", "region"]).size().unstack(fill_value=0).reindex(range(24), fill_value=0)
    f, ax = plt.subplots(figsize=(9, 4))
    for col, color in zip(tab.columns, [AZUL, NARANJA, VERDE]):
        ax.plot(tab.index, tab[col], marker="o", ms=3, color=color, label=col)
    ax.set_xlabel("Hora del día"); ax.set_ylabel("Salidas programadas"); ax.legend(title="Región")
    ax.set_title("Actividad aérea durante el día"); ax.set_xticks(range(0, 24, 2))
    return _guardar(f, "04_actividad_por_hora")


def fig_mapa(v, ok, la, lo):
    """Mapa simple: vuelos (rojo = atípico), aeropuertos y rutas simuladas."""
    f, ax = plt.subplots(figsize=(9, 6))
    for i in range(len(la)):
        ax.plot(lo[i], la[i], color=GRIS, lw=.7, alpha=.5)
    atip = (v.get("atip_speed", 0) == 1) | (v.get("atip_alt", 0) == 1)
    ax.scatter(v.loc[~atip, "lng"], v.loc[~atip, "lat"], s=6, color=AZUL, label="Vuelos activos")
    ax.scatter(v.loc[atip, "lng"], v.loc[atip, "lat"], s=40, color=ROJO, marker="x", label="Atípicos (velocidad/altitud)")
    for i, (n, la_, lo_) in AEROPUERTOS.items():
        ax.scatter(lo_, la_, s=60, color=NARANJA, edgecolor="k", zorder=3); ax.annotate(i, (lo_, la_), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.plot([], [], color=GRIS, label="Rutas simuladas (GPU)")
    ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud"); ax.legend(loc="lower left"); ax.set_title("Vuelos, atípicos y rutas simuladas")
    return _guardar(f, "05_mapa_vuelos")


def fig_thingspeak(ts):
    """Serie de tiempo de las variables del sensor ThingSpeak."""
    cols = [c for c in ts.columns if c not in ("created_at", "fuente") and ts[c].notna().any()][:4]
    f, ax = plt.subplots(len(cols), 1, figsize=(10, 2 * len(cols) + 1), sharex=True)
    for a, c, color in zip(np.atleast_1d(ax), cols, [AZUL, NARANJA, VERDE, ROJO]):
        a.plot(ts["created_at"], ts[c], color=color, lw=1); a.set_ylabel(c[:18], fontsize=8)
    np.atleast_1d(ax)[0].set_title(f"Sensor ThingSpeak ({ts['fuente'].iloc[0]})")
    return _guardar(f, "06_thingspeak")


def fig_correlacion(corr):
    """Mapa de calor de correlaciones (divergente: azul negativo, rojo positivo)."""
    f, ax = plt.subplots(figsize=(8, 6.5))
    im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.index); ax.grid(False)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
    f.colorbar(im, label="Spearman"); ax.set_title("Correlación clima / calidad del aire / retrasos")
    return _guardar(f, "07_correlacion")


def fig_rendimiento(tiempos_mpi, bench, t_async):
    """Resumen de paralelismo: asyncio, MPI y GPU."""
    f, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].bar(["Secuencial", "asyncio"], t_async, color=[GRIS, AZUL]); ax[0].set_ylabel("segundos")
    ax[0].set_title("12 peticiones simuladas (0.5 s c/u)")
    ks = sorted(tiempos_mpi, key=int)
    ax[1].bar([f"{k} proc." for k in ks], [tiempos_mpi[k] for k in ks], color=NARANJA)
    ax[1].set_title("Bootstrap con MPI"); ax[1].set_ylabel("segundos")
    if bench.get("gpu_disponible"):
        ax[2].bar(["CPU NumPy", "GPU (con copias)", "GPU (solo kernel)"],
                  [bench["cpu_s"], bench["gpu_total_s"], bench["gpu_kernel_s"]], color=[GRIS, VERDE, AZUL])
        ax[2].set_yscale("log"); ax[2].set_title(f"Haversine, {bench['n']:,} pares"); ax[2].set_ylabel("segundos (log)")
    else:
        ax[2].text(.5, .5, "Sin GPU en este entorno", ha="center"); ax[2].set_axis_off()
    return _guardar(f, "08_rendimiento")


In [ ]:
%%writefile reporte_pdf.py
"""
reporte_pdf.py
==============
Genera el REPORTE PDF del proyecto de forma DINÁMICA: todo número, tabla y figura
sale de la ejecución actual. Si cambian las API keys, los datos o el código y
vuelves a ejecutar el notebook, el PDF se actualiza solo.

Se usa la librería ReportLab (Platypus): armamos una lista de "flowables"
(párrafos, tablas, imágenes) y `SimpleDocTemplate.build` los acomoda en páginas.

Entrada: un diccionario `ctx` que arma la última celda del notebook con los
resultados de cada fase (ver la celda "9. Reporte PDF").
"""
import datetime
import os

from PIL import Image as PILImage
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import (Image, Paragraph, Preformatted, SimpleDocTemplate,
                                Spacer, Table, TableStyle)

AZUL = colors.HexColor("#0072B2")
REPO = "https://github.com/Anngel-o/trafico-aereo-computo-paralelo"
COLAB = "https://colab.research.google.com/drive/1a8DVBhEwQmpc0ToJ4koVQ1SSI4rfTell?usp=sharing"
EQUIPO = ["Jose Angel Ortiz Meraz (353195)", "Luz Mariam Garcia Castillo (348409)", "Carlos Sánchez Fuentes (368058)"]
_ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1", parent=_ss["Heading1"], fontSize=15, spaceBefore=10, textColor=colors.HexColor("#0b3d6b"))
H2 = ParagraphStyle("H2", parent=_ss["Heading2"], fontSize=12, spaceBefore=8, textColor=AZUL)
B = ParagraphStyle("B", parent=_ss["BodyText"], fontSize=10, leading=14)
CAP = ParagraphStyle("CAP", parent=B, fontSize=8.5, textColor=colors.grey)
CELDA = ParagraphStyle("CELDA", parent=B, fontSize=8.5, leading=10.5)
CODE = ParagraphStyle("CODE", fontName="Courier", fontSize=7.8, leading=9.6,
                      backColor=colors.HexColor("#f3f5f7"), borderPadding=4, leftIndent=4)
OUT = ParagraphStyle("OUT", parent=CODE, backColor=colors.HexColor("#eef6ee"))


def _p(texto, estilo=B):
    return Paragraph(texto, estilo)


def _esc(t):
    """Escapa caracteres especiales para que ReportLab no los tome como etiquetas."""
    return str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def _codigo(t):
    return [Preformatted(t.strip("\n"), CODE), Spacer(1, 6)]


def _salida(t):
    return [_p("<b>Salida de esta ejecución:</b>", CAP), Preformatted(t.strip("\n"), OUT), Spacer(1, 6)]


def _figura(ruta, pie, ancho_cm=16.5):
    """Inserta una imagen conservando su proporción; si no existe, lo avisa."""
    if not os.path.exists(ruta):
        return [_p(f"(figura no disponible: {ruta})", CAP)]
    im = PILImage.open(ruta)
    return [Image(ruta, width=ancho_cm * cm, height=ancho_cm * cm * im.height / im.width),
            _p(pie, CAP), Spacer(1, 6)]


def _tabla(filas, anchos=None):
    """Tabla con encabezado azul. Las celdas de texto largo se envuelven en Paragraph."""
    filas = [[c if i == 0 else (_p(_esc(c), CELDA) if isinstance(c, str) and len(c) > 28 else c)
              for c in fila] for i, fila in enumerate(filas)]
    t = Table(filas, colWidths=anchos, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), AZUL), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("GRID", (0, 0), (-1, -1), .3, colors.lightgrey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f3f5f7")])]))
    return [t, Spacer(1, 8)]


def _f(x, d=2):
    return f"{x:.{d}f}"


def generar_reporte(ctx, ruta="Reporte_Proyecto_CPD_Trafico_Aereo.pdf", dir_figuras="figuras"):
    """
    Arma el PDF completo. `ctx` debe traer:
      datos (dict de DataFrames), vuelos, horarios, tabla, reg (resumen por región),
      mpi_df, tiempos_mpi, bench, t_async (t_secuencial, t_asyncio), respuestas,
      atipicos (dict), ingesta_s, gpu_nombre, rutas_shape.
    Devuelve la ruta del PDF.
    """
    d, h, t = ctx["datos"], ctx["horarios"], ctx["tabla"]
    fuentes = {k: str(v["fuente"].iloc[0]) for k, v in d.items()}
    simuladas = [k for k, f in fuentes.items() if f == "SIMULADO"]
    reales = [k for k, f in fuentes.items() if f != "SIMULADO"]
    tm, bench = ctx["tiempos_mpi"], ctx["bench"]
    t_sec, t_as = ctx["t_async"]
    S = []

    # ---- Portada -----------------------------------------------------------
    S += [_p("Análisis de tráfico aéreo y condiciones ambientales usando sensores y APIs en la nube",
             ParagraphStyle("T", parent=_ss["Title"], fontSize=19, leading=23)),
          _p("Proyecto 3ª Evaluación · Cómputo paralelo y distribuido (CPD)",
             ParagraphStyle("s", parent=B, alignment=1, fontSize=11)),
          _p("<b>Equipo:</b> " + " · ".join(EQUIPO), ParagraphStyle("eq", parent=B, alignment=1, fontSize=9.5)),
          _p(f'<b>Repositorio:</b> <link href="{REPO}" color="blue">{REPO}</link>', ParagraphStyle("rp", parent=B, alignment=1, fontSize=8.5)),
          _p(f'<b>Colab:</b> <link href="{COLAB}" color="blue">{COLAB}</link>', ParagraphStyle("cb", parent=B, alignment=1, fontSize=8)),
          _p(f"Generado automáticamente el {datetime.datetime.now():%Y-%m-%d %H:%M} · "
             f"GPU: {ctx['gpu_nombre']}",
             ParagraphStyle("s2", parent=B, alignment=1, fontSize=9, textColor=colors.grey)),
          Spacer(1, 10)]

    # ---- Aviso dinámico sobre el origen de los datos -------------------------
    if simuladas:
        S += [_p("Aviso importante sobre los datos", H1),
              _p(f"En esta ejecución usaron datos <b>REALES</b>: {', '.join(reales) or 'ninguna fuente'}. "
                 f"Usaron datos <b>SIMULADOS</b> (modo demo, columna <i>fuente = SIMULADO</i>): <b>{', '.join(simuladas)}</b>, "
                 "porque falta su API key o la API no respondió. En los datos simulados el retraso depende por diseño de la lluvia, "
                 "la visibilidad y el viento, así que esas correlaciones <b>validan el pipeline, no son un hallazgo real</b>. "
                 "Al agregar las keys en Secretos de Colab y volver a ejecutar, este reporte se regenera con datos reales.")]
    else:
        S += [_p("Origen de los datos", H1),
              _p("Todas las fuentes de esta ejecución son <b>reales</b> (AirLabs, OpenWeather, OpenAQ y ThingSpeak).")]

    # ---- 1. Objetivo y arquitectura -----------------------------------------
    S += [_p("1. Objetivo y arquitectura", H1),
          _p("Aplicar cómputo paralelo para relacionar el tráfico aéreo en México con las condiciones ambientales. "
             "Arquitectura: <b>fuentes en la nube -> asyncio -> MPI4Py -> CUDA/CuPy/Numba -> dashboard</b>, con Pandas como apoyo.")]
    S += _tabla([["Fase", "Tecnología", "Paralelismo", "Qué hace aquí"],
                 ["1. Ingesta", "asyncio + aiohttp", "Concurrencia (E/S de red)", "Pide a 4 APIs ~40 peticiones a la vez en un solo hilo"],
                 ["2. Limpieza", "pandas", "Vectorización", "Tipos, duplicados, banderas de retraso, unión clima + retrasos"],
                 ["3. Distribución", "mpi4py (OpenMPI)", "Procesos (memoria distribuida)", "Reparte aeropuertos; bootstrap del retraso medio"],
                 ["4. Aceleración", "Numba CUDA + CuPy", "Miles de hilos en GPU", "Distancias, atípicos, rutas simuladas, correlaciones"],
                 ["5. Dashboard", "matplotlib", "-", "Figuras y respuestas a las 5 preguntas"]],
                [2.6 * cm, 3.4 * cm, 4 * cm, 7.4 * cm])

    # ---- 2. Requisitos previos: cuentas, keys y verificación ------------------
    S += [_p("2. Requisitos previos: cuentas y API keys", H1),
          _p("Antes de pedir datos reales hay un paso previo: <b>crear una cuenta en cada servicio y generar su API key</b>. "
             "Las llaves se guardan en Colab (Secretos, con 'Acceso desde el cuaderno' activado) y <b>nunca se escriben en el código</b> ni se suben a GitHub.")]
    S += _tabla([["Servicio", "¿Cuenta y key?", "Cómo se obtiene", "Secreto en Colab"],
                 ["AirLabs", "Sí", "Registro en airlabs.co; la key aparece en el panel de la cuenta (formato UUID)", "AIRLABS_KEY"],
                 ["OpenWeather", "Sí", "Registro en openweathermap.org; API keys en la cuenta (32 hexadecimales). Una key nueva puede tardar un rato en activarse", "OPENWEATHER_KEY"],
                 ["OpenAQ v3", "Sí", "Registro en explore.openaq.org/register; la key está en explore.openaq.org/account y se manda en el header X-API-Key", "OPENAQ_KEY"],
                 ["ThingSpeak", "<b>No</b>", "Canal público 12397 (estación meteorológica de MathWorks): ThingSpeak permite leer canales públicos con un GET sin autenticación y de forma gratuita", "(no aplica)"]],
                [2.4 * cm, 2.4 * cm, 9.4 * cm, 3.2 * cm])
    S += [_p("<b>Por qué ThingSpeak no necesita key:</b> las API keys de ThingSpeak (Read/Write) solo hacen falta para leer un canal <i>privado</i> o para "
             "<i>escribir</i> datos desde sensores propios. Aquí solo se leen datos ambientales públicos (temperatura, humedad, presión, viento), "
             "por eso basta con el número de canal (12397) y la URL <i>https://api.thingspeak.com/channels/12397/feeds.json</i>.")]
    S += [_p("Verificación automática de las keys (módulo verificar_claves.py)", H2),
          _p("Antes de ingestar, el sistema comprueba para cada key que <b>exista</b>, tenga el <b>formato</b> esperado (detecta llaves copiadas incompletas o con espacios) "
             "y que la API la <b>acepte</b> con una petición real de prueba. La llave nunca se imprime completa.")]
    cl = ctx.get("claves")
    if cl is not None:
        S += _tabla([["Servicio", "Llave (enmascarada)", "Formato", "Estado", "Detalle"]] +
                    [[r.servicio, r.llave, r.formato, r.estado, r.detalle] for r in cl.itertuples()],
                    [2.3 * cm, 3.4 * cm, 1.9 * cm, 2.7 * cm, 7.1 * cm])
    else:
        S += [_p("(La verificación no se ejecutó en esta corrida.)")]

    # ---- 3. Fuentes ----------------------------------------------------------
    S += [_p("3. Fuentes de datos de esta ejecución", H1)]
    S += _tabla([["Dato", "Filas", "Fuente"]] + [[k, len(v), fuentes[k]] for k, v in d.items()], [4 * cm, 3 * cm, 8 * cm])

    # ---- 3. Paso a paso -----------------------------------------------------
    S += [_p("4. Ejecución paso a paso", H1)]
    S += [_p("Paso 1 · Ingesta concurrente con asyncio (ingesta_async.py)", H2),
          _p("Pedir datos a una API es esperar a la red. Con <b>async/await</b> y <b>asyncio.gather</b> se lanzan todas las peticiones juntas; "
             "un <b>Semaphore</b> limita cuántas van a la vez y si una API falla se reintenta o se cae a modo demo.")]
    S += _codigo("datos = await ingesta_async.ingestar_todo()")
    S += _salida(f"Ingesta terminada en {_f(ctx['ingesta_s'])} s\n" +
                 "\n".join(f"  {k:11s}: {len(v):5d} filas | fuente = {fuentes[k]}" for k, v in d.items()) +
                 f"\n\nDemostración (12 peticiones de 0.5 s): secuencial {_f(t_sec)} s | asyncio {_f(t_as)} s "
                 f"-> {_f(t_sec / t_as, 1)}x más rápido")

    v, hh = ctx["vuelos"], ctx["horarios"]
    S += [_p("Paso 2 · Limpieza con pandas (limpieza_pandas.py)", H2),
          _p("Conversión a números, eliminación de duplicados y registros sin posición, banderas <i>cancelado</i> y <i>retrasado</i> (>15 min), "
             "hora de salida, región por latitud y unión (<b>groupby + merge</b>) para tener una fila por aeropuerto.")]
    S += _salida(f"Vuelos en el aire: {len(v)} | Salidas programadas: {len(hh)} | Cancelados: {int(hh.cancelado.sum())} "
                 f"| Retrasados >15 min: {int(hh.retrasado.sum())}")
    reg = ctx["reg"].reset_index()
    S += _tabla([["Región", "Vuelos activos", "Velocidad media (km/h)", "Altitud media (m)"]] +
                [[str(r.region), int(r.vuelos), _f(r.velocidad_media_kmh, 1), _f(r.altitud_media_m, 1)] for r in reg.itertuples()],
                [4 * cm, 3.5 * cm, 4.5 * cm, 4 * cm])

    S += [_p("Paso 3 · Distribución con MPI4Py (mpi_analisis.py)", H2),
          _p("<b>mpirun</b> lanza varias copias del script (<i>ranks</i>, cada una con su memoria). Rank 0 hace <b>bcast</b> del parámetro, "
             "<b>scatter</b> de los aeropuertos, cada proceso calcula un <b>bootstrap</b> del retraso medio y rank 0 junta con <b>gather</b> y <b>Reduce</b>.")]
    ks = sorted(tm, key=int)
    S += _tabla([["Procesos", "Tiempo total (s)", "Speedup vs 1 proceso"]] +
                [[k, _f(tm[k]), _f(tm["1"] / tm[k]) + "x"] for k in ks], [4 * cm, 4 * cm, 5 * cm])
    S += [_p("Colab ofrece pocos CPU virtuales (normalmente 2): con más procesos que núcleos no hay más aceleración y el arranque de procesos pesa. "
             "Lo importante es el patrón scatter/gather, que en un clúster sí escala.")]
    m = ctx["mpi_df"]
    S += _tabla([["Aeropuerto", "n", "Retraso medio", "IC95 inf", "IC95 sup", "Calculado por rank"]] +
                [[r.iata, int(r.n), _f(r.media, 1), _f(r.ic95_inf, 1), _f(r.ic95_sup, 1), int(r.rank_que_calculo)] for r in m.itertuples()])

    S += [_p("Paso 4 · GPU con Numba CUDA y CuPy (gpu_numba.py)", H2),
          _p("Un <b>kernel</b> es una función que ejecutan miles de hilos a la vez; cada hilo usa <b>cuda.grid()</b> para saber qué dato le toca. "
             "Kernels: <b>k_haversine</b> (distancia km), <b>k_zscore</b> (atípicos con |z|>3) y <b>k_ruta</b> (trayectoria de círculo máximo en malla 2D). "
             "CuPy calcula media, desviación y la matriz de correlación en GPU.")]
    a = ctx["atipicos"]
    S += _salida(f"GPU disponible: {bench['gpu_disponible']} ({ctx['gpu_nombre']})\n"
                 f"Atípicos de velocidad: {a['speed']} | de altitud: {a['alt']} | de retraso: {a['retraso']}\n"
                 f"Rutas simuladas: {ctx['rutas_shape']} (vuelos x pasos)")
    if bench["gpu_disponible"]:
        S += _tabla([[f"Haversine con {bench['n']:,} pares", "Tiempo (s)", "Speedup"],
                     ["CPU (NumPy)", _f(bench["cpu_s"], 4), "1.0x"],
                     ["GPU incluyendo copias CPU<->GPU", _f(bench["gpu_total_s"], 4), _f(bench["speedup_total"], 1) + "x"],
                     ["GPU solo kernel", _f(bench["gpu_kernel_s"], 4), _f(bench["speedup_kernel"], 1) + "x"]],
                    [8 * cm, 3.5 * cm, 3.5 * cm])
        S += [_p(f"Error máximo CPU vs GPU: {bench['max_error_km'] * 1000:.1f} m (la GPU usa float32). Las copias de memoria CPU<->GPU "
                 "cuestan tiempo, por eso la GPU conviene con muchos datos; la primera llamada compila el kernel (JIT), por eso hay un calentamiento.")]
    else:
        S += [_p("Esta ejecución no tuvo GPU: se usó NumPy. Activa T4 GPU en el tipo de entorno para ver la aceleración.")]
    S += _figura(f"{dir_figuras}/08_rendimiento.png", "Figura 1. Paralelismo en las tres capas: asyncio, MPI y GPU.")

    S += [_p("Paso 5 · Visualización", H2)]
    for n, pie, w in [("01_actividad_vuelos", "Figura 2. Vuelos por región, aerolínea y rutas más frecuentes.", 16.5),
                      ("02_retrasos_aeropuerto", "Figura 3. Retraso medio por aeropuerto con IC 95% (MPI).", 12),
                      ("05_mapa_vuelos", "Figura 4. Vuelos, atípicos (X rojas, GPU) y rutas simuladas (kernel CUDA).", 13),
                      ("04_actividad_por_hora", "Figura 5. Salidas por hora del día y región.", 13),
                      ("03_clima_vs_retraso", "Figura 6. Retraso medio contra lluvia, visibilidad y viento.", 16.5),
                      ("07_correlacion", "Figura 7. Correlación de Spearman (cp.corrcoef en GPU).", 12),
                      ("06_thingspeak", f"Figura 8. Sensor ThingSpeak ({fuentes['thingspeak']}).", 14)]:
        S += _figura(f"{dir_figuras}/{n}.png", pie, w)

    # ---- 6. Respuestas -------------------------------------------------------
    S += [_p("7. Respuestas a las preguntas de análisis", H1)]
    if simuladas:
        S += [_p("<b>Recordatorio:</b> parte de los datos es simulada; estas respuestas muestran el método, no conclusiones reales.")]
    for r in ctx["respuestas"]:
        S += [_p(_esc(r)), Spacer(1, 4)]
    top = t.sort_values("retraso_medio", ascending=False).head(6)
    S += [_p("Aeropuertos con más retraso:")]
    S += _tabla([["IATA", "Salidas", "Retraso medio (min)", "% >15 min", "Cancelados", "Lluvia (mm/h)", "Visibilidad (m)"]] +
                [[x.iata, int(x.n_vuelos), _f(x.retraso_medio, 1), _f(x.pct_retrasados, 0), int(x.cancelados),
                  _f(x.lluvia_mm, 1), _f(x.visibilidad_m, 0)] for x in top.itertuples()])

    # ---- 7. Conclusiones ----------------------------------------------------
    sp_mpi = max(tm["1"] / tm[k] for k in tm)
    S += [_p("8. Conclusiones y limitaciones", H1),
          _p(f"&bull; <b>asyncio</b>: {_f(t_sec / t_as, 1)}x menos espera en la demostración, porque las peticiones se traslapan.<br/>"
             f"&bull; <b>MPI</b>: mejor aceleración medida {_f(sp_mpi)}x, limitada por los núcleos de Colab y la poca carga.<br/>"
             + (f"&bull; <b>GPU</b>: {_f(bench['speedup_total'], 1)}x con copias y {_f(bench['speedup_kernel'], 1)}x solo cálculo con {bench['n']:,} datos.<br/>"
                if bench["gpu_disponible"] else "&bull; <b>GPU</b>: no disponible en esta ejecución.<br/>") +
             f"&bull; Con {len(t)} aeropuertos las correlaciones tienen poca potencia estadística: son indicios, no causalidad. El clima es el del "
             "momento de la consulta, no el de cada retraso.<br/>"
             "&bull; El plan gratuito de AirLabs limita campos y consultas; conviene guardar y acumular datos en el tiempo.")]
    S += [_p("9. Glosario de conceptos clave", H1),
          _p("<b>asyncio:</b> concurrencia (no paralelismo) para esperar a la red. <b>MPI:</b> procesos con memoria separada que se mandan mensajes (scatter, gather, Reduce). "
             "<b>CUDA:</b> kernel = función que corren miles de hilos; cuda.grid() da el índice; hay que copiar datos CPU<->GPU. "
             "<b>Atípico (z-score):</b> a más de 3 desviaciones estándar de la media. <b>Bootstrap:</b> remuestrear con reemplazo para estimar incertidumbre. "
             "<b>Haversine:</b> distancia entre dos puntos de una esfera.")]

    SimpleDocTemplate(ruta, pagesize=letter, leftMargin=2 * cm, rightMargin=2 * cm, topMargin=1.8 * cm,
                      bottomMargin=1.8 * cm, title="Análisis de tráfico aéreo y condiciones ambientales").build(S)
    return ruta


In [ ]:
%%writefile verificar_claves.py
"""
verificar_claves.py
===================
AUDITORÍA de las API keys antes de usarlas.

Problema que resuelve: si una llave está mal copiada (espacio extra, letra
faltante), es de otro servicio o todavía no se activó, las APIs responden con
error y el sistema caería en silencio a datos simulados. Este módulo lo detecta
y lo dice claramente, sin imprimir nunca la llave completa.

Para CADA servicio hace 3 comprobaciones:
  1. EXISTE   -> la llave está en Secretos de Colab (o variable de entorno).
  2. FORMATO  -> tiene la forma esperada (ej. OpenWeather = 32 caracteres
                 hexadecimales; AirLabs = UUID). Un formato raro suele ser una
                 llave copiada incompleta o de otro servicio.
  3. FUNCIONA -> se hace UNA petición real y barata a la API y se interpreta la
                 respuesta (200 = válida, 401/403 = rechazada, 429 = límite...).

ThingSpeak se prueba SIN llave porque el canal 12397 es público (ver config.py).
"""
import asyncio
import re

import aiohttp
import pandas as pd

from config import THINGSPEAK_CANAL, obtener_clave

# nombre del secreto -> (servicio, patrón esperado, descripción del formato)
SERVICIOS = {
    "AIRLABS_KEY": ("AirLabs", re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.I),
                    "UUID (8-4-4-4-12 hexadecimales)"),
    "OPENWEATHER_KEY": ("OpenWeather", re.compile(r"^[0-9a-f]{32}$", re.I), "32 caracteres hexadecimales"),
    "OPENAQ_KEY": ("OpenAQ", re.compile(r"^[0-9a-f]{64}$", re.I), "64 caracteres hexadecimales"),
}


def enmascarar(clave):
    """Muestra solo el inicio y el final de la llave para poder compararla sin exponerla."""
    if not clave:
        return "-"
    return f"{clave[:4]}…{clave[-2:]} ({len(clave)} car.)" if len(clave) > 8 else "(muy corta)"


def revisar_formato(nombre, cruda):
    """
    Devuelve (clave_limpia, formato_ok, observaciones).
    Detecta espacios/saltos de línea/comillas invisibles y formato inesperado.
    """
    obs = []
    if not cruda:
        return None, False, ["no está configurada"]
    limpia = cruda.strip().strip("\"'").strip()
    if limpia != cruda:
        obs.append("tenía espacios/saltos de línea/comillas (se limpian automáticamente)")
    _, patron, descripcion = SERVICIOS[nombre]
    ok = bool(patron.match(limpia))
    if not ok:
        obs.append(f"formato inesperado: se esperaba {descripcion}")
    return limpia, ok, obs


# ---------------------------------------------------------------------------
# Interpretación de respuestas HTTP (función pura -> se puede probar sin red)
# ---------------------------------------------------------------------------
def clasificar(estado_http, cuerpo=None):
    """
    Traduce la respuesta de una API a (estado, detalle).
    estado_http: código HTTP (None si no hubo conexión). cuerpo: JSON si lo hay.
    """
    if estado_http is None:
        return "SIN CONEXIÓN", "no se pudo contactar a la API"
    if isinstance(cuerpo, dict) and cuerpo.get("error"):       # AirLabs: HTTP 200 + "error"
        err = cuerpo["error"]
        msg = err.get("message", err) if isinstance(err, dict) else err
        return "INVÁLIDA", f"la API la rechazó: {msg}"
    if estado_http == 200:
        return "OK", "la API la aceptó"
    if estado_http in (401, 403):
        return "INVÁLIDA", f"HTTP {estado_http}: llave incorrecta, mal copiada o aún sin activar"
    if estado_http == 429:
        return "LÍMITE", "HTTP 429: se agotó la cuota/límite de peticiones (la llave sí es válida)"
    return "ERROR", f"HTTP {estado_http}"


async def _get(sesion, url, params=None, headers=None):
    """GET que devuelve (código_http, json_o_None) sin lanzar excepciones."""
    try:
        async with sesion.get(url, params=params, headers=headers,
                              timeout=aiohttp.ClientTimeout(total=20)) as r:
            try:
                cuerpo = await r.json(content_type=None)
            except Exception:
                cuerpo = None
            return r.status, cuerpo
    except Exception:
        return None, None


async def _probar(sesion, nombre, clave):
    """Una petición real y barata a la API del servicio."""
    if nombre == "AIRLABS_KEY":
        return await _get(sesion, "https://airlabs.co/api/v9/airports", {"api_key": clave, "iata_code": "CUU"})
    if nombre == "OPENWEATHER_KEY":
        return await _get(sesion, "https://api.openweathermap.org/data/2.5/weather",
                          {"lat": 28.70, "lon": -105.96, "appid": clave})
    if nombre == "OPENAQ_KEY":
        return await _get(sesion, "https://api.openaq.org/v3/locations", {"limit": 1}, {"X-API-Key": clave})


async def verificar_todo(mostrar=True):
    """
    Revisa las 3 llaves + el canal público de ThingSpeak.
    Devuelve un DataFrame con: servicio, llave (enmascarada), formato, estado, detalle.
    """
    filas = []
    async with aiohttp.ClientSession() as s:
        for nombre, (servicio, _, _) in SERVICIOS.items():
            cruda = obtener_clave(nombre, limpiar=False)
            limpia, formato_ok, obs = revisar_formato(nombre, cruda)
            if not limpia:
                filas.append(dict(servicio=servicio, secreto=nombre, llave="-", formato="-",
                                  estado="SIN KEY", detalle="no configurada -> esta fuente usará datos SIMULADOS"))
                continue
            http, cuerpo = await _probar(s, nombre, limpia)
            estado, detalle = clasificar(http, cuerpo)
            if obs:
                detalle += " | " + "; ".join(obs)
            filas.append(dict(servicio=servicio, secreto=nombre, llave=enmascarar(limpia),
                              formato="correcto" if formato_ok else "REVISAR", estado=estado, detalle=detalle))

        # ThingSpeak: canal público, sin llave
        http, cuerpo = await _get(s, f"https://api.thingspeak.com/channels/{THINGSPEAK_CANAL}/feeds.json", {"results": 1})
        estado, detalle = clasificar(http, cuerpo)
        if estado == "OK" and isinstance(cuerpo, dict):
            detalle = f"canal público «{cuerpo.get('channel', {}).get('name', '?')}» accesible sin llave"
        filas.append(dict(servicio="ThingSpeak", secreto="(sin llave: canal público 12397)", llave="no aplica",
                          formato="no aplica", estado=estado, detalle=detalle))

    df = pd.DataFrame(filas)
    if mostrar:
        icono = {"OK": "✅", "INVÁLIDA": "❌", "SIN KEY": "⚪", "LÍMITE": "⚠️", "SIN CONEXIÓN": "🔌", "ERROR": "❌"}
        print("VERIFICACIÓN DE API KEYS")
        for r in df.itertuples():
            print(f" {icono.get(r.estado, '?')} {r.servicio:11s} | llave: {r.llave:22s} | formato: {r.formato:9s} | {r.estado}: {r.detalle}")
        malas = df[df.estado.isin(["INVÁLIDA", "ERROR"])]
        if len(malas):
            print("\n -> Revisa las llaves marcadas ❌: copia de nuevo la llave completa desde el sitio del servicio, "
                  "sin espacios, y confirma el nombre exacto del secreto y el interruptor 'Acceso desde el cuaderno'.")
    return df


if __name__ == "__main__":
    asyncio.run(verificar_todo())


## 1b. Paso previo – Verificación de las API keys
Antes de descargar datos se comprueba cada llave: **existe**, tiene el **formato** correcto y la API la **acepta** (petición real de prueba). La llave nunca se imprime completa. ThingSpeak no necesita llave: el canal 12397 es público.

In [ ]:
import verificar_claves
claves = await verificar_claves.verificar_todo()
claves

## 2. Paso 1 – Ingesta concurrente con `asyncio`
Primero una demostración sencilla del concepto (12 peticiones simuladas de 0.5 s) y luego la descarga real.

In [ ]:
import asyncio, os, json, numpy as np, pandas as pd
import ingesta_async, limpieza_pandas as L, analisis as A, gpu_numba as G, visualizacion as V
os.makedirs("datos", exist_ok=True); os.makedirs("resultados", exist_ok=True)

t_sec, t_async = await ingesta_async.demo_asyncio_vs_secuencial()
print(f"Secuencial: {t_sec:.2f} s   |   asyncio: {t_async:.2f} s   ->  {t_sec/t_async:.1f}x más rápido")

In [ ]:
# Descarga REAL de todas las APIs al mismo tiempo (en Colab se puede usar `await` directo)
datos = await ingesta_async.ingestar_todo()
for nombre, df in datos.items():
    df.to_csv(f"datos/{nombre}.csv", index=False)   # los guardamos: MPI los leerá de aquí
datos["clima"].round(1).head()

## 3. Paso 2 – Limpieza y análisis tabular con `pandas`

In [ ]:
vuelos = L.limpiar_vuelos(datos["vuelos"])
horarios = L.limpiar_horarios(datos["horarios"])
print(f"Vuelos en el aire: {len(vuelos)} | Salidas programadas: {len(horarios)} "
      f"| Cancelados: {horarios.cancelado.sum()} | Retrasados >15 min: {horarios.retrasado.sum()}")
resumen = L.resumen_por_aeropuerto(horarios, datos["clima"], datos["aire"])
resumen.round(1)

In [ ]:
# Velocidad/altitud promedio y vuelos activos por región
A.resumen_vuelos(vuelos)

## 4. Paso 3 – Procesamiento distribuido con `MPI4Py`
`mpirun` lanza varias copias de `mpi_analisis.py`. Rank 0 reparte los aeropuertos (`scatter`), cada proceso calcula el **bootstrap** del retraso medio y rank 0 junta todo (`gather`/`Reduce`). Lo corremos con 1, 2 y 4 procesos para medir el tiempo (Colab tiene solo 2 CPU, así que más de 2 no acelera).

In [ ]:
!rm -f resultados/mpi_tiempos.json
for n in (1, 2, 4):
    print(f"\n================ mpirun con {n} proceso(s) ================")
    !mpirun --allow-run-as-root --oversubscribe -n {n} python mpi_analisis.py 200000 2>&1 | grep -E "rank|TOTAL" 

In [ ]:
mpi_df = pd.read_csv("resultados/mpi_estadisticas.csv")
tiempos_mpi = json.load(open("resultados/mpi_tiempos.json"))
print("Speedup MPI:", {k: round(tiempos_mpi["1"]/v, 2) for k, v in tiempos_mpi.items()})
mpi_df.round(1)

## 5. Paso 4 – Aceleración con GPU (`Numba CUDA` + `CuPy`)
Kernels propios: distancia haversine, z-score de atípicos y trayectorias de ruta. CuPy: media/desviación y correlaciones.

In [ ]:
print("GPU disponible:", G.HAY_GPU)
vuelos, horarios = A.marcar_atipicos(vuelos, horarios)          # kernel z-score
print("Atípicos ->", vuelos.filter(like="atip_").sum().to_dict(), "| retrasos:", int(horarios.atip_retraso.sum()))
display(vuelos[(vuelos.atip_speed == 1) | (vuelos.atip_alt == 1)][["flight_iata", "speed", "alt", "lat", "lng"]].head())

cercanos = A.vuelos_cerca_de_aeropuertos(vuelos)                # kernel haversine (vuelos x aeropuertos)
tabla = A.tabla_final(resumen, cercanos)
ok, rut_lat, rut_lon = A.simular_rutas(vuelos)                  # kernel de rutas (malla 2D)
print("Rutas simuladas:", rut_lat.shape, "(vuelos x pasos)")

In [ ]:
bench = G.benchmark(5_000_000)
print(bench)

In [ ]:
cols = A.VARS_CLIMA + ["pm25", "retraso_medio", "pct_retrasados", "vuelos_cercanos"]
corr = G.matriz_correlacion(tabla, cols, "spearman")            # cp.corrcoef en GPU
corr.round(2)

## 6. Paso 5 – Visualización (Dashboard con Matplotlib)

In [ ]:
import matplotlib.pyplot as plt
rutas = A.rutas_mas_frecuentes(vuelos)
V.fig_actividad_vuelos(vuelos, rutas); plt.show()
V.fig_retrasos_aeropuerto(tabla, mpi_df); plt.show()
V.fig_clima_vs_retraso(tabla); plt.show()
V.fig_actividad_hora(horarios, A.region_de_lat); plt.show()
V.fig_mapa(vuelos, ok, rut_lat, rut_lon); plt.show()
V.fig_thingspeak(datos["thingspeak"]); plt.show()
V.fig_correlacion(corr); plt.show()
V.fig_rendimiento(tiempos_mpi, bench, (t_sec, t_async)); plt.show()

## 7. Respuestas a las preguntas de análisis

In [ ]:
respuestas = A.responder_preguntas(tabla, horarios)
for r in respuestas: print(r, "\n")
fuentes = {k: v["fuente"].iloc[0] for k, v in datos.items()}
print("Fuentes de datos usadas:", fuentes)
json.dump({"respuestas": respuestas, "fuentes": fuentes, "benchmark": bench, "mpi": tiempos_mpi},
          open("resultados/resumen.json", "w"), ensure_ascii=False, indent=1, default=float)

## 8. Conclusiones y limitaciones
* **asyncio** reduce el tiempo de espera de red porque las peticiones se traslapan.
* **MPI** reparte trabajo independiente por aeropuerto; la aceleración está limitada por los 2 núcleos de Colab.
* **GPU**: gana cuando hay millones de datos; con pocos datos las copias CPU↔GPU dominan.
* Con 12 aeropuertos las correlaciones tienen poco poder estadístico: son indicios, no prueba causal. El clima se toma *en el momento de la consulta*, no en el instante de cada retraso.
* Si alguna fuente aparece como `SIMULADO`, esas conclusiones solo validan el pipeline, no el mundo real.

## 9. Reporte PDF dinámico
Esta celda arma el PDF con **los resultados de esta ejecución** (tablas, tiempos, figuras y respuestas). Si cambias keys, datos o código y vuelves a ejecutar todo, el PDF se actualiza solo.

In [ ]:
!pip install -q reportlab
import reporte_pdf
from numba import cuda
ctx = dict(
    claves=claves, datos=datos, vuelos=vuelos, horarios=horarios, tabla=tabla, reg=A.resumen_vuelos(vuelos),
    mpi_df=mpi_df, tiempos_mpi=tiempos_mpi, bench=bench, t_async=(t_sec, t_async), respuestas=respuestas,
    atipicos=dict(speed=int(vuelos.atip_speed.sum()), alt=int(vuelos.atip_alt.sum()), retraso=int(horarios.atip_retraso.sum())),
    ingesta_s=ingesta_async.ULTIMA_INGESTA_S, rutas_shape=rut_lat.shape,
    gpu_nombre=(str(cuda.get_current_device().name, "utf-8") if G.HAY_GPU and isinstance(cuda.get_current_device().name, bytes) else cuda.get_current_device().name if G.HAY_GPU else "sin GPU (NumPy)"))
ruta_pdf = reporte_pdf.generar_reporte(ctx)
print("PDF generado:", ruta_pdf)
try:                                   # descarga automática a tu computadora
    from google.colab import files
    files.download(ruta_pdf)
except Exception as e:
    print("Descárgalo desde el panel de archivos (carpeta izquierda):", e)